# Golden Record Relationships — Level 2

Explore how saved Level 1 educational resource records relate to other educational objects. Use Oak source data to identify and model relationships such as lesson-to-unit, sequence, assets, quizzes, and supporting materials while preserving source-native evidence separately from normalized relationship types.

In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Project paths
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SAMPLE_DIR = PROJECT_ROOT / "samples" / "golden_records" / "oak"

# Environment
load_dotenv(PROJECT_ROOT / ".env")

R2_BUCKET = os.getenv("R2_BUCKET_NAME")
OAK_API_KEY = os.getenv("OAK_API_KEY")

# Project imports
from collectors.shared.storage import get_r2_client

print("Project root:", PROJECT_ROOT)
print("Sample dir:", SAMPLE_DIR)
print("R2 bucket configured:", bool(R2_BUCKET))
print("Oak API key configured:", bool(OAK_API_KEY))

Project root: /home/nunto/dev/rubric-agent
Sample dir: /home/nunto/dev/rubric-agent/samples/golden_records/oak
R2 bucket configured: True
Oak API key configured: True


In [2]:
level_1_files = sorted(SAMPLE_DIR.glob("*.json"))

level_1_records = []

for path in level_1_files:
    with path.open(encoding="utf-8") as f:
        level_1_records.append(json.load(f))

print(f"Loaded Level 1 records: {len(level_1_records)}")

for record in level_1_records:
    print(
        record["source"]["source_id"],
        "->",
        record["identity"]["title"],
    )

Loaded Level 1 records: 6
combine-multiplication-with-addition-and-subtraction -> Combine multiplication with addition and subtraction
compose-tangram-images -> Compose tangram images
draw-polygons-specified-by-coordinates-in-the-first-quadrant -> Draw polygons specified by coordinates in the first quadrant
explain-the-size-of-a-part-in-relation-to-the-whole -> Explain the size of a part in relation to the whole
explore-recognise-and-compare-three-different-3d-shapes -> Explore, recognise and compare three different 3D shapes
solve-problems-involving-missing-coordinates -> Solve problems involving missing coordinates


In [5]:
for record in level_1_records:
    print(
        record["source"]["source_id"],
        "->",
        record["identity"].get("unit_title"),
        "\n"
    )

combine-multiplication-with-addition-and-subtraction -> Order of operations 

compose-tangram-images -> Recognise, compose, decompose and manipulate 2D and 3D shapes 

draw-polygons-specified-by-coordinates-in-the-first-quadrant -> Coordinates 

explain-the-size-of-a-part-in-relation-to-the-whole -> Unit fractions as part of a whole 

explore-recognise-and-compare-three-different-3d-shapes -> Recognise, compose, decompose and manipulate 2D and 3D shapes 

solve-problems-involving-missing-coordinates -> Area, perimeter, position and direction 



In [11]:
import os
import sys
import json
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from collectors.shared.storage import get_r2_client

load_dotenv(PROJECT_ROOT / ".env")

R2_BUCKET = os.getenv("R2_BUCKET_NAME")
OAK_API_KEY = os.getenv("OAK_API_KEY")

r2 = get_r2_client()

# Oak bulk metadata currently stored in R2
OAK_BULK_KEY = (
    "metadata/oak/math/primary/"
    "resource_19d7cfd594eb4ef3.json"
)

response = r2.get_object(
    Bucket=R2_BUCKET,
    Key=OAK_BULK_KEY,
)

oak_data = json.load(response["Body"])

print(type(oak_data))

if isinstance(oak_data, dict):
    print(oak_data.keys())

<class 'dict'>
dict_keys(['sequenceSlug', 'subjectTitle', 'sequence', 'lessons'])


In [12]:
# Inspect possible Level 2 relationship fields from the original Oak records
oak_lessons = oak_data["lessons"]
sample_slugs = {
    record["source"]["source_id"]
    for record in level_1_records
}

sample_oak_lessons = [
    lesson
    for lesson in oak_lessons
    if lesson.get("lessonSlug") in sample_slugs
]

relationship_terms = (
    "unit",
    "order",
    "position",
    "sequence",
    "programme",
    "lesson",
)

print(f"Matched Oak lessons: {len(sample_oak_lessons)}")

for lesson in sample_oak_lessons:
    print("\n" + "=" * 80)
    print(lesson.get("lessonSlug"))
    print(lesson.get("lessonTitle"))
    print("-" * 80)

    for key in sorted(lesson.keys()):
        if any(term in key.lower() for term in relationship_terms):
            print(f"{key}: {lesson.get(key)}")

Matched Oak lessons: 6

explore-recognise-and-compare-three-different-3d-shapes
Explore, recognise and compare three different 3D shapes
--------------------------------------------------------------------------------
lessonKeywords: [{'keyword': 'Similar', 'description': 'When one thing is like something else.'}, {'keyword': 'Cone', 'description': 'A 3D shape with one vertex, one circular face and one curved surface.'}, {'keyword': 'Sphere', 'description': 'A 3D shape with one curved surface.'}, {'keyword': 'Cylinder', 'description': 'A 3D shape with two circular faces and one curved surface.'}]
lessonSlug: explore-recognise-and-compare-three-different-3d-shapes
lessonTitle: Explore, recognise and compare three different 3D shapes
pupilLessonOutcome: I can recognise, name and compare three different 3D shapes.
unitSlug: recognise-compose-decompose-and-manipulate-2d-and-3d-shapes
unitTitle: Recognise, compose, decompose and manipulate 2D and 3D shapes

compose-tangram-images
Compose ta

In [ ]:
# Inspect possible Level 2 relationship fields
# for the six saved sample lessons

sample_slugs = {
    record["source"]["source_id"]
    for record in level_1_records
}

sample_oak_lessons = [
    lesson
    for lesson in oak_lessons
    if lesson.get("lessonSlug") in sample_slugs
]

relationship_terms = (
    "unit",
    "order",
    "position",
    "sequence",
    "programme",
)

print(f"Matched Oak lessons: {len(sample_oak_lessons)}")

for lesson in sample_oak_lessons:
    print("\n" + "=" * 80)
    print(lesson.get("lessonSlug"))
    print(lesson.get("lessonTitle"))
    print("-" * 80)

    for key in sorted(lesson.keys()):
        if any(term in key.lower() for term in relationship_terms):
            print(f"{key}: {lesson.get(key)}")

## Compare Level 1 records with Oak API enrichment